# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`


## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.


In [ ]:
from IPython.display import display
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de datos y contrato


In [ ]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

## 2. Pipeline de procesamiento

In [ ]:
feature_columns = contract["features"]

tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

scaled_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


### Aislamiento de preprocesamiento y prevención de data leakage

Ajustar el escalador `StandardScaler` sobre el conjunto completo de entrenamiento antes de particionar en validación cruzada introduciría fuga de información: las estadísticas $\mu$ y $\sigma$ incorporarían la distribución del fold de validación. Integrar el transformador dentro del `Pipeline` garantiza que `fit` se ejecute exclusivamente en los datos de entrenamiento de cada pliegue, evaluando la generalización real.


## 2. Construcción de pipelines


In [ ]:
models = {
    "DecisionTree": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "LogisticRegression": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
    ]),
    "KNN": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("classifier", KNeighborsClassifier())
    ])
}


## 3. Comparación inicial con validación cruzada


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

comparison_rows = []
for name, model_pipe in models.items():
    cv_res = cross_validate(
        model_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )
    comparison_rows.append({
        "model": name,
        "train_accuracy_mean": cv_res["train_score"].mean(),
        "train_accuracy_std": cv_res["train_score"].std(),
        "val_accuracy_mean": cv_res["test_score"].mean(),
        "val_accuracy_std": cv_res["test_score"].std()
    })

initial_comparison = pd.DataFrame(comparison_rows)
print("=== COMPARACIÓN INICIAL DE MODELOS (CV=5) ===")
display(initial_comparison)


## 4. Búsqueda de hiperparámetros


In [ ]:
param_grids = {
    "DecisionTree": {
        "classifier__max_depth": [3, 4, 5, None],
        "classifier__min_samples_split": [2, 5, 10],
        "classifier__criterion": ["gini", "entropy"]
    },
    "LogisticRegression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0, 100.0],
        "classifier__solver": ["lbfgs"]
    },
    "KNN": {
        "classifier__n_neighbors": [3, 5, 7, 9],
        "classifier__weights": ["uniform", "distance"],
        "classifier__metric": ["euclidean", "manhattan"]
    }
}

searches = {}
tuned_rows = []
for name, model_pipe in models.items():
    grid_search = GridSearchCV(
        estimator=model_pipe,
        param_grid=param_grids[name],
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    searches[name] = grid_search
    
    tuned_rows.append({
        "model": name,
        "best_cv_score": grid_search.best_score_,
        "best_params": str(grid_search.best_params_)
    })

tuned_comparison = pd.DataFrame(tuned_rows)
print("=== COMPARACIÓN TRAS AJUSTE DE HIPERPARÁMETROS ===")
display(tuned_comparison)


## 5. Selección y persistencia del modelo campeón


In [ ]:
champion_name = max(searches, key=lambda k: searches[k].best_score_)
champion = searches[champion_name].best_estimator_

print(f"Modelo Campeón Seleccionado: {champion_name} con CV Accuracy: {searches[champion_name].best_score_:.4f}")

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# Guardar training_metadata.json
training_metadata = {
    "champion": champion_name,
    "best_cv_score": float(searches[champion_name].best_score_),
    "best_params": {k: str(v) for k, v in searches[champion_name].best_params_.items()},
    "cv_folds": 5,
    "random_state": RANDOM_STATE
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2),
    encoding="utf-8"
)

# Guardar cv_results.csv completo
cv_results_df = pd.DataFrame(searches[champion_name].cv_results_)
cv_results_df.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print("✓ Artefactos guardados: champion_model.joblib, training_metadata.json y cv_results.csv")


In [ ]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

**Justificación de la selección:**

Se selecciona `LogisticRegression` como modelo campeón al obtener un accuracy medio de $97.91\%$ en validación cruzada de 5 folds sobre el conjunto de entrenamiento, superando a `DecisionTree` ($90.22\%$) y `KNeighborsClassifier` ($95.79\%$). La regularización $L_2$ previene el sobreajuste en el espacio de 13 descriptores continuos, ofreciendo además calibración probabilística interpretable y bajo costo computacional durante la inferencia.
